# 02_mlp_pytorch.ipynb

Ce notebook reproduit le MLP simple en PyTorch pour montrer autograd, Module, DataLoader, et le training loop standard.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
np.random.seed(0)
torch.manual_seed(0)


## Préparer les données (mêmes que dans le notebook NumPy)


In [ ]:
digits = load_digits()
X = digits.data / 16.0
y = digits.target
mask = y < 3
X = X[mask]
y = y[mask]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
# convertir en tenseurs
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)


## Définir le modèle PyTorch (équivalent au MLP NumPy)


In [ ]:
class TorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = TorchMLP(input_dim=X_train.shape[1], hidden_dim=64, output_dim=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


## Training loop standard
On GPU : placer le modèle et les tenseurs sur device='cuda' si disponible. Ici on montre la version CPU (compatible GPU si tu changes le device).


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
epochs = 30
losses = []
for epoch in range(epochs):
    running_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    losses.append(epoch_loss)
    if epoch % 5 == 0:
        print(f'Epoch {epoch}, loss={epoch_loss:.4f}')
plt.plot(losses)
plt.title('Training loss (PyTorch MLP)')
plt.show()

# Eval
model.eval()
with torch.no_grad():
    logits_test = model(X_test_t.to(device))
    preds = logits_test.argmax(dim=1).cpu().numpy()
    acc = (preds == y_test).mean()
    print('Accuracy PyTorch MLP:', acc)


---
Ce notebook montre comment transposer l'implémentation NumPy en PyTorch. Les étapes suivantes proposées :
- ajouter des visualisations de gradients (normes) pendant l'entraînement
- comparer comportements (loss et accuracy) entre NumPy et PyTorch sur le même seed
- expérimenter avec batchnorm / different optimizers (Adam)
